# Notebook 4 — AMPK Triad Simulation (Master Model)
### AMPK Signaling Case Study: Metformin at the Triad

This is the flagship simulation tying the whole case study together — a Python/Colab port of the
original interactive dose-response tool. One control (AMPK activation intensity, driven by
metformin) drives three tissue panels simultaneously:

- **Hepatocyte** (diabetes) — monotonic benefit, plateauing
- **Tumor cell** (cancer) — monotonic benefit, preclinical evidence only
- **Trophoblast** (preeclampsia) — biphasic: a therapeutic window with harm on either side

The other three notebooks each dug into one dimension in more depth (time, statistical inference,
population heterogeneity). This notebook is the compact, presentation-ready summary — good for a
live demo during your seminar talk.


In [ ]:
!pip install ipywidgets -q
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider


### Model definitions (shared across all three tissues)

In [ ]:
COLORS = dict(liver='#1F6F6B', tumor='#6B3F6E', placenta='#B15A2E',
              good='#3F7D5C', caution='#B9860C', harm='#9B3A32', neutral='#8A8F86')

def liver_state(x):
    ampk = x
    mtor = np.clip(100 - 0.55*x, 8, 100)
    glucose = np.clip(100 - 0.8*x, 14, 100)
    if x < 20: status, kind = 'Subtherapeutic', 'neutral'
    elif x <= 90: status, kind = 'Effective glycemic control', 'good'
    else: status, kind = 'Plateau -- diminishing returns', 'neutral'
    return dict(ampk=ampk, mtor=mtor, outcome=glucose, outcome_label='Hepatic glucose output',
                status=status, kind=kind)

def tumor_state(x):
    ampk = x
    mtor = np.clip(100 - 0.7*x, 5, 100)
    prolif = np.clip(100 - 100*x/(x+25), 8, 100)
    if x < 25: status, kind = 'Minimal antiproliferative signal', 'neutral'
    else: status, kind = 'Antiproliferative signal (preclinical)', 'good'
    return dict(ampk=ampk, mtor=mtor, outcome=prolif, outcome_label='Proliferation signal',
                status=status, kind=kind)

def placenta_state(x):
    ampk = np.clip(55 + 0.45*x, 0, 100)
    mtor = np.clip(100 - 0.65*x, 5, 100)
    sflt1 = 90 - 60*np.sin(np.pi*np.clip(x,0,100)/100) + 25*(np.clip(x,0,100)/100)**4
    if x < 15: status, kind = 'Insufficient -- baseline dysfunction persists', 'neutral'
    elif x <= 65: status, kind = 'Therapeutic window -- restoring balance', 'good'
    else: status, kind = 'Caution -- compounding mTORC1 suppression', 'harm'
    return dict(ampk=ampk, mtor=mtor, outcome=sflt1, outcome_label='sFlt-1 (anti-angiogenic)',
                status=status, kind=kind)

TISSUES = [
    ('Hepatocyte (diabetes)', liver_state, COLORS['liver']),
    ('Tumor cell (cancer)', tumor_state, COLORS['tumor']),
    ('Trophoblast (preeclampsia)', placenta_state, COLORS['placenta']),
]


### Panel renderer

In [ ]:
def status_color(kind):
    return COLORS.get(kind, COLORS['neutral'])

def render_triad(x):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
    for ax, (title, fn, color) in zip(axes, TISSUES):
        s = fn(x)
        labels = ['AMPK\nactivation', 'mTORC1\nactivity', s['outcome_label']]
        values = [s['ampk'], s['mtor'], s['outcome']]
        bar_colors = [color, COLORS['neutral'], status_color(s['kind']) if labels[2]==s['outcome_label'] else color]
        bars = ax.barh(labels, values, color=[color, COLORS['neutral'], status_color(s['kind'])])
        ax.set_xlim(0, 130)
        for b, v in zip(bars, values):
            ax.text(min(v,128)+2, b.get_y()+b.get_height()/2, f'{v:.0f}', va='center', fontsize=10)
        ax.set_title(title, color=color, fontsize=13, fontweight='bold')
        ax.text(0, -0.9, s['status'], transform=ax.transData, fontsize=10.5,
                color=status_color(s['kind']), fontweight='bold')
        ax.invert_yaxis()
    plt.suptitle(f'AMPK activation intensity (metformin exposure): {x}/100', fontsize=13, y=1.04)
    plt.tight_layout()
    plt.show()

    if x < 15:
        note = ("At this level, hepatic and tumor effects are minimal, and the trophoblast remains "
                 "close to its untreated preeclamptic state.")
    elif x <= 65:
        note = ("Interesting middle zone: liver and tumor outcomes keep improving, and the "
                 "trophoblast is now inside its therapeutic window -- all three tissues currently benefit.")
    elif x <= 85:
        note = ("Liver and tumor readouts continue to improve or plateau, but the trophoblast has "
                 "passed its optimum -- mTORC1 suppression is now compounding the disease mechanism.")
    else:
        note = ("At maximal activation, liver and tumor outcomes sit at their plateau (no harm, just "
                 "diminishing returns), while the trophoblast has overshot its window entirely.")
    print(note)


### Static snapshots at key dose levels

In [ ]:
for x in [0, 40, 55, 90]:
    render_triad(x)


### Interactive explorer (the main event)
Drag the slider and watch all three tissues respond simultaneously.

In [ ]:
@interact(x=IntSlider(min=0, max=100, step=1, value=0, description='AMPK dose'))
def explore(x=0):
    render_triad(x)


### How this ties the whole case study together

| | Hepatocyte | Tumor cell | Trophoblast |
|---|---|---|---|
| Shape of curve | Monotonic, plateauing | Monotonic, plateauing | Biphasic, therapeutic window |
| Evidence strength | Strong, established | Strong signal, weak RCTs (see Notebook 2) | Emerging (see Notebooks 1 & 3) |
| What determines the outcome | Direct suppression of gluconeogenesis | mTORC1-driven growth suppression | Baseline AMPK/mTORC1 imbalance already present |

Use this notebook as your live demo centerpiece, Notebook 1 to discuss *timing* of effects, Notebook
2 to discuss *why the cancer evidence is contested*, and Notebook 3 to discuss *why a single optimal
dose is an idealization* in a real, heterogeneous placenta.
